In [ ]:
#@title Prevent disconnections
%%html
<audio src="https://oobabooga.github.io/silence.m4a" controls>

In [ ]:
#@title ComfyUI Setup
!apt install aria2

# Install pinggy for the public tunnel
!pip install pinggy -q

# Clone ComfyUI (everything stays in the Colab instance, no Drive)
!git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git /content/ComfyUI

# Install ComfyUI Manager
%cd /content/ComfyUI/custom_nodes
!git clone --depth 1 https://github.com/Comfy-Org/ComfyUI-Manager.git

# Install dependencies (Colab already ships a CUDA-enabled torch)
%cd /content/ComfyUI
!pip install -r requirements.txt -q


In [ ]:
#@title Download Upscaler

USER_AGENT = '"User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64)"'

UPSCALER_PATH = "/content/ComfyUI/models/upscale_models"

UPSCALER = {
    "file": "4x-UltraSharp.safetensors",
    "url": "https://huggingface.co/Kim2091/UltraSharp/resolve/main/4x-UltraSharp.safetensors"
}

!mkdir -p $UPSCALER_PATH
!aria2c --enable-http-keep-alive=false --header=$USER_AGENT --console-log-level=error -c -x 16 -s 16 -k 1M --summary-interval=5 -d $UPSCALER_PATH -o "{UPSCALER['file']}" "{UPSCALER['url']}"

In [ ]:
#@title Select Model
KEY = ""  # @param {type:"string"}
KEY = "token=" + KEY.strip()

MODELS = {
    "cyberrealisticPony_semiRealV40": {
        "file": "cyberrealisticPony_semiRealV40.safetensors",
        "url": f"https://civitai.com/api/download/models/2268768?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}",
    },
    "deepDarkHentaiMixNSFW_v61Hybrid": {
        "file": "deepDarkHentaiMixNSFW_v61Hybrid.safetensors",
        "url": f"https://civitai.com/api/download/models/634653?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}",
    },
    "novaCartoon_v10": {
        "file": "novaCartoon_v10.safetensors",
        "url": f"https://civitai.com/api/download/models/821389?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}",
    },
}

CHOICE = "cyberrealisticPony_semiRealV40"  # @param ["cyberrealisticPony_semiRealV40", "deepDarkHentaiMixNSFW_v61Hybrid", "novaCartoon_v10"]

MODEL = MODELS[CHOICE]
MODEL_PATH = "/content/ComfyUI/models/checkpoints"


In [ ]:
#@title Download Model
USER_AGENT = '"User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64)"'

!mkdir -p $MODEL_PATH
!aria2c --enable-http-keep-alive=false --header=$USER_AGENT --console-log-level=error -c -x 16 -s 16 -k 1M --summary-interval=5 -d $MODEL_PATH -o "{MODEL['file']}" "{MODEL['url']}"


In [ ]:
#@title LAUNCH!

# Open a public tunnel to ComfyUI with pinggy and print the URL
import pinggy

tunnel1 = pinggy.start_tunnel(
    forwardto="localhost:8188",
)

print(f"Tunnel started - URLs:\n{tunnel1.urls}")


# Run ComfyUI, listening on all network interfaces
%cd /content/ComfyUI

!python main.py --listen 0.0.0.0 --cache-none --mmap-torch-files --reserve-vram 2 --fast-disk
